# Etapa 2 — Pré-processamento para detecção de intrusão

**Inteligência Artificial · Ciência da Computação · PUC Minas · Trabalho Prático 3**

O objetivo é medir o efeito do pré-processamento sobre a classificação de conexões em **normal** e **anomaly**. A árvore de decisão e uma rede neural pequena funcionam como referências para comparar as técnicas. A comparação dos três classificadores fica para a Etapa 3.

No Google Colab, escolha **Ambiente de execução → Executar tudo** e, se solicitado, selecione o **treino.csv oficial do Canvas**. O notebook contém todo o código. Em uma sessão nova, as dependências podem levar um tempo para instalar e os 137 ajustes levam alguns minutos em CPU. O teste oficial fica reservado para as previsões finais.

In [ ]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys
import tempfile
from hashlib import sha256
from statistics import mean, stdev
from IPython.display import Markdown, display

candidatos = [
    Path("data/raw/treino.csv"),
    Path("treino.csv"),
    Path("/content/treino.csv"),
]
treino_csv = next((arquivo for arquivo in candidatos if arquivo.is_file()), None)
if treino_csv is None:
    try:
        from google.colab import files
    except ImportError as erro:
        raise FileNotFoundError(
            "Coloque o treino.csv oficial na pasta inicial ou em data/raw e execute novamente."
        ) from erro
    treino_csv = Path("data/raw/treino.csv")
    treino_csv.parent.mkdir(parents=True, exist_ok=True)
    print("Selecione o treino.csv oficial do Canvas.")
    files.upload_file(str(treino_csv))

treino_csv = treino_csv.resolve()
with treino_csv.open(encoding="utf-8-sig", newline="") as arquivo:
    cabecalho = next(csv.reader(arquivo), [])
obrigatorias = {"class", "protocol_type", "service", "flag", "src_bytes", "total_bytes"}
if not obrigatorias.issubset(cabecalho):
    raise ValueError(
        f"O arquivo em {treino_csv} não tem o cabeçalho do treino oficial. "
        "Substitua-o pelo treino.csv correto e execute novamente."
    )
trabalho = Path(tempfile.mkdtemp(prefix="ia-etapa2-"))
codigo_dados = trabalho / "dados_etapa2.py"
codigo_tecnicas = trabalho / "tecnicas_etapa2.py"
codigo_experimentos = trabalho / "executar_etapa2.py"
print("Base encontrada:", treino_csv.name)

## Ambiente de execução

As versões usadas nos experimentos são verificadas antes de iniciar o cálculo. Quando necessário, o notebook instala as dependências em uma pasta temporária e inicia um processo Python separado, de modo que a execução use essas versões mesmo se o Colab já tiver carregado outras bibliotecas.

In [ ]:
VERSOES = {
    "numpy": "2.5.4",
    "pandas": "3.0.1",
    "scipy": "1.18.1",
    "scikit-learn": "1.7.2",
    "imbalanced-learn": "0.14.1",
    "joblib": "1.6.0",
    "threadpoolctl": "3.8.0",
    "sklearn-compat": "0.1.6",
}

def consultar_versoes(ambiente):
    consulta = (
        "import importlib.metadata as m, json, sys; "
        "print(json.dumps({n: next((d.version for d in m.distributions(name=n)), None) "
        "for n in sys.argv[1:]}))"
    )
    resposta = subprocess.check_output(
        [sys.executable, "-c", consulta, *VERSOES], env=ambiente, text=True
    )
    return json.loads(resposta)

ambiente = os.environ.copy()
ambiente["PYTHONIOENCODING"] = "utf-8"
ambiente["PYTHONNOUSERSITE"] = "1"
ambiente["OPENBLAS_NUM_THREADS"] = "1"
ambiente["OMP_NUM_THREADS"] = "1"
if consultar_versoes(ambiente) != VERSOES:
    assinatura = sha256(json.dumps(VERSOES, sort_keys=True).encode()).hexdigest()[:12]
    bibliotecas = Path(tempfile.gettempdir()) / f"ia-etapa2-{sys.implementation.cache_tag}-{assinatura}"
    caminho_existente = ambiente.get("PYTHONPATH")
    ambiente["PYTHONPATH"] = os.pathsep.join(filter(None, (str(bibliotecas), caminho_existente)))
    if consultar_versoes(ambiente) != VERSOES:
        print("Instalando as versões dos experimentos. Aguarde a conclusão desta célula.", flush=True)
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
             "--only-binary=:all:", "--upgrade", "--target", str(bibliotecas),
             *[f"{nome}=={versao}" for nome, versao in VERSOES.items()]],
            check=True, env={**os.environ, "PYTHONNOUSERSITE": "1"},
        )
versoes_ativas = consultar_versoes(ambiente)
if versoes_ativas != VERSOES:
    raise RuntimeError(f"As versões do ambiente não coincidiram com as previstas: {versoes_ativas}")
print("Python:", sys.version.split()[0])
print("Bibliotecas:", versoes_ativas)

## Dados e protocolo de validação

O diagnóstico da Etapa 1 motivou a retirada experimental da coluna constante e das três entradas cuja disponibilidade no momento da predição ainda precisa ser confirmada. As versões A, B e C retêm 45, 41 e 40 atributos, respectivamente. A preserva as cinco derivadas, B conserva apenas o total de bytes entre as derivadas e C retira as cinco. As categorias são padronizadas com remoção de espaços e de diferenças de caixa. Duração e bytes negativos e taxas fora de [0, 1] são tratados como ausências nas cópias de trabalho. Valores extremos plausíveis e o código 2 de su_attempted são preservados.

Criamos grupos com atributos iguais na versão C, sem usar o rótulo. Cada grupo fica inteiro no treino ou na validação. Há quatro divisões para comparar alternativas e uma quinta, excluída de todos esses ajustes, para conferir combinações predefinidas. Essa quinta divisão havia participado das médias do piloto anterior. A composição de cada validação é preservada. Toda decisão que aprende valores, como a imputação ou a amostragem, usa apenas a parcela de ajuste da rodada.

In [ ]:
%%writefile "{codigo_dados}"
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

ALVO = "class"
COLUNAS_EXCLUIDAS_EXPERIMENTOS = (
    "num_outbound_cmds", "tempo_resposta_analista_min", "tipo_alerta_soc", "score_ids_legado",
)
SEMENTE = 42
NOMINAIS = ["protocol_type", "service", "flag", "protocolo_cod"]
COPIAS = ["src_bytes_kb", "dst_bytes_kb", "same_srv_pct", "protocolo_cod"]


def carregar_treino(caminho: str | Path) -> pd.DataFrame:
    """Lê o CSV oficial e valida a coluna alvo."""
    caminho = Path(caminho)
    if not caminho.is_file():
        raise FileNotFoundError(f"Base oficial não encontrada: {caminho}")
    dados = pd.read_csv(caminho)
    if ALVO not in dados or dados.empty:
        raise ValueError(f"A base deve ser não vazia e conter {ALVO!r}.")
    if dados[ALVO].isna().any() or not set(dados[ALVO].unique()).issubset({"normal", "anomaly"}):
        raise ValueError("A coluna class deve conter somente normal ou anomaly, sem ausências.")
    return dados.copy()


def identificar_taxas(dados: pd.DataFrame) -> list[str]:
    """Retorna apenas frações ``*_rate``; ``*_pct`` usa escala percentual."""
    return [c for c in dados.select_dtypes(include="number") if c.lower().endswith("rate")]


def preparar_base(dados):
    entradas = dados.drop(columns=["class", *COLUNAS_EXCLUIDAS_EXPERIMENTOS])
    for coluna in NOMINAIS:
        entradas[coluna] = entradas[coluna].astype("string").str.strip().str.casefold().replace("", pd.NA)

    invalidos = {}
    for coluna in ("duration", "src_bytes", "dst_bytes"):
        invalidos[coluna] = entradas[coluna].lt(0)
    for coluna in identificar_taxas(entradas):
        invalidos[coluna] = entradas[coluna].notna() & ~entradas[coluna].between(0, 1)
    invalidos["src_bytes_kb"] = invalidos["src_bytes"] | entradas["src_bytes_kb"].lt(0)
    invalidos["dst_bytes_kb"] = invalidos["dst_bytes"] | entradas["dst_bytes_kb"].lt(0)
    invalidos["total_bytes"] = invalidos["src_bytes"] | invalidos["dst_bytes"] | entradas["total_bytes"].lt(0)
    percentual = entradas["same_srv_pct"]
    invalidos["same_srv_pct"] = invalidos["same_srv_rate"] | (percentual.notna() & ~percentual.between(0, 100))
    for coluna, mascara in invalidos.items():
        entradas[coluna] = entradas[coluna].mask(mascara)

    representacoes = {
        "A": entradas,
        "B": entradas.drop(columns=COPIAS),
        "C": entradas.drop(columns=[*COPIAS, "total_bytes"]),
    }
    assert [m.shape[1] for m in representacoes.values()] == [45, 41, 40]
    grupos = representacoes["C"].groupby(list(representacoes["C"]), dropna=False, sort=False).ngroup()
    assert grupos.notna().all()
    return representacoes, grupos


def escolher_linhas(x, y, indices, regra):
    if regra == "todas":
        return indices
    chaves = x.iloc[indices].assign(rotulo=y[indices])
    mascara = np.ones(len(indices), dtype=bool)
    if regra in {"sem_conflitos", "sem_ambos"}:
        grupos = chaves.groupby(list(x), dropna=False, sort=False)["rotulo"]
        mascara &= grupos.transform("nunique").to_numpy() == 1
    if regra in {"sem_duplicatas", "sem_ambos"}:
        mascara &= ~chaves.duplicated().to_numpy()
    return indices[mascara]


def separar_divisoes(representacoes, y, grupos):
    divisor = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
    particoes = list(divisor.split(representacoes["C"], y, grupos))
    reserva = particoes[-1][1]
    divisoes = []
    for fold, (treino, validacao) in enumerate(particoes, 1):
        if fold != 5:
            treino = np.setdiff1d(treino, reserva)
            assert set(grupos.iloc[treino]).isdisjoint(grupos.iloc[reserva])
            assert set(grupos.iloc[validacao]).isdisjoint(grupos.iloc[reserva])
        assert set(grupos.iloc[treino]).isdisjoint(grupos.iloc[validacao])
        divisoes.append((fold, treino, validacao))
    return divisoes

## Técnicas comparadas

- **Dados inconsistentes e redundantes (Partes 2 e 3).** Comparamos manter as linhas, retirar duplicatas de mesmo rótulo, retirar grupos conflitantes e fazer as duas retiradas. Na imputação numérica, variamos mediana, média, mediana com indicador e KNN de cinco vizinhos. O KNN usa 2.000 referências amostradas da parcela de ajuste para limitar o custo. Para as ausências nominais, comparamos categoria explícita e moda.
- **Conversão simbólico-numérica (Parte 4).** Comparamos one-hot completo, one-hot agrupando serviços vistos menos de 20 vezes no ajuste e códigos inteiros. A codificação inteira atribui uma ordem artificial, enquanto o one-hot usa mais colunas. A tabela mostra esse custo.
- **Transformação numérica (Parte 6).** Comparamos sem escala, Z-score, Min-Max e escala robusta em uma rede de 16 neurônios, com até 16 iterações. A rede é sensível à escala, enquanto a árvore tende a preservar suas decisões sob mudanças monotônicas de escala.
- **Balanceamento (Parte 1).** RandomOverSampler e RandomUnderSampler usam a mesma proporção de uma anomalia para dois registros normais. A validação mantém a proporção original.
- **Redução (Parte 7).** Comparamos manter todas as colunas, selecionar 60 por um filtro univariado e projetar em 60 componentes por PCA.

O critério principal é **F1-macro**, que dá o mesmo peso às duas classes. Também registramos o recall e a precisão de anomaly, a quantidade de colunas e as linhas usadas no ajuste.

In [ ]:
%%writefile "{codigo_tecnicas}"
import warnings
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
import numpy as np
from sklearn import config_context
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, RobustScaler, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from dados_etapa2 import NOMINAIS, SEMENTE, escolher_linhas


def transformar_numericas(treino, validacao, metodo, escala):
    treino = treino.to_numpy(dtype=np.float64, na_value=np.nan)
    validacao = validacao.to_numpy(dtype=np.float64, na_value=np.nan)
    if metodo == "mediana_indicador":
        tem_ausencia = np.isnan(treino).any(axis=0)
        indicadores_treino = np.isnan(treino[:, tem_ausencia])
        indicadores_val = np.isnan(validacao[:, tem_ausencia])
    if metodo == "knn":
        # Limita o custo de vizinhança a uma amostra sem rótulos da parte de treino.
        normalizador = StandardScaler().fit(treino)
        amostra = np.random.default_rng(SEMENTE).choice(len(treino), size=min(2000, len(treino)), replace=False)
        imputador = KNNImputer(n_neighbors=5, keep_empty_features=True)
        imputador.fit(normalizador.transform(treino[amostra]))
        with config_context(working_memory=48):
            treino = normalizador.inverse_transform(imputador.transform(normalizador.transform(treino)))
            validacao = normalizador.inverse_transform(imputador.transform(normalizador.transform(validacao)))
    else:
        estrategia = "mean" if metodo == "media" else "median"
        imputador = SimpleImputer(strategy=estrategia, keep_empty_features=True)
        treino, validacao = imputador.fit_transform(treino), imputador.transform(validacao)
    normalizadores = {"zscore": StandardScaler, "robusta": RobustScaler, "minmax": MinMaxScaler}
    if escala in normalizadores:
        normalizador = normalizadores[escala]()
        treino, validacao = normalizador.fit_transform(treino), normalizador.transform(validacao)
    if metodo == "mediana_indicador":
        treino = np.column_stack((treino, indicadores_treino))
        validacao = np.column_stack((validacao, indicadores_val))
    return treino, validacao


def transformar_categoricas(treino, validacao, metodo, codificacao):
    estrategia = "most_frequent" if metodo == "moda" else "constant"
    imputador = SimpleImputer(strategy=estrategia, fill_value="__ausente__")
    colunas = list(treino)
    treino = imputador.fit_transform(treino.to_numpy(dtype=object, na_value=np.nan))
    validacao = imputador.transform(validacao.to_numpy(dtype=object, na_value=np.nan))
    if codificacao == "inteiros":
        codificador = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        return codificador.fit_transform(treino), codificador.transform(validacao)
    cod_treino, cod_val = [], []
    for i, coluna in enumerate(colunas):
        agrupar = codificacao == "onehot_raros" and coluna == "service"
        codificador = OneHotEncoder(
            sparse_output=False,
            min_frequency=20 if agrupar else None,
            handle_unknown="infrequent_if_exist" if agrupar else "ignore",
        )
        cod_treino.append(codificador.fit_transform(treino[:, i:i + 1]))
        cod_val.append(codificador.transform(validacao[:, i:i + 1]))
    return np.column_stack(cod_treino), np.column_stack(cod_val)


def transformar(x, y, indices, validacao, config):
    indices = escolher_linhas(x, y, indices, config.linhas)
    treino, validacao = x.iloc[indices], x.iloc[validacao]
    categoricas = [c for c in NOMINAIS if c in x]
    numericas = x.columns.drop(categoricas).tolist()
    num_treino, num_val = transformar_numericas(treino[numericas], validacao[numericas], config.numericos, config.escala)
    cat_treino, cat_val = transformar_categoricas(treino[categoricas], validacao[categoricas], config.nominais, config.codificacao)
    return np.column_stack((num_treino, cat_treino)), np.column_stack((num_val, cat_val)), y[indices]


def medir(x_treino, x_val, y_treino, y_val, config):
    if config.reducao != "nenhuma":
        k = min(60, x_treino.shape[1])
        if config.reducao == "filtro":
            redutor = SelectKBest(f_classif, k=k)
        else:
            redutor = PCA(n_components=k, svd_solver="randomized", random_state=SEMENTE)
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message=r"Features .* are constant.")
            warnings.filterwarnings("ignore", message="invalid value encountered in divide", category=RuntimeWarning)
            x_treino = redutor.fit_transform(x_treino, y_treino)
        x_val = redutor.transform(x_val)
    antes = len(y_treino)
    anomalias_antes = int(np.sum(y_treino == "anomaly"))
    amostradores = {"over": RandomOverSampler, "under": RandomUnderSampler}
    if config.balanceamento in amostradores:
        amostrador = amostradores[config.balanceamento](sampling_strategy=0.5, random_state=SEMENTE)
        x_treino, y_treino = amostrador.fit_resample(x_treino, y_treino)
    assert x_treino.shape[1] == x_val.shape[1]
    assert np.isfinite(x_treino).all() and np.isfinite(x_val).all()
    if config.modelo == "arvore":
        modelo = DecisionTreeClassifier(max_depth=8, random_state=SEMENTE)
    else:
        modelo = MLPClassifier(
            hidden_layer_sizes=(16,), random_state=SEMENTE, max_iter=16,
            batch_size=512, n_iter_no_change=16, early_stopping=False,
        )
    with warnings.catch_warnings(record=True) as avisos:
        warnings.simplefilter("always", ConvergenceWarning)
        modelo.fit(x_treino, y_treino)
    previsto = modelo.predict(x_val)
    return {
        "f1_macro": f1_score(y_val, previsto, average="macro"),
        "recall_anomaly": recall_score(y_val, previsto, pos_label="anomaly", zero_division=0),
        "precisao_anomaly": precision_score(y_val, previsto, pos_label="anomaly", zero_division=0),
        "colunas": x_treino.shape[1],
        "linhas_antes_amostragem": antes,
        "anomalias_antes_amostragem": anomalias_antes,
        "linhas_ajuste": len(y_treino),
        "anomalias_ajuste": int(np.sum(y_treino == "anomaly")),
        "atingiu_limite_iteracoes": any(issubclass(a.category, ConvergenceWarning) for a in avisos),
    }

## Comparações e execução

A referência usa a versão C, mediana, categoria explícita de ausência, one-hot completo, Z-score, todas as linhas, nenhum balanceamento e nenhuma redução. Primeiro mudamos uma técnica por vez. Depois avaliamos as mesmas combinações nas quatro divisões iniciais e na quinta. As configurações da quinta divisão foram fixadas antes de observar seu resultado nesta rodada.

In [ ]:
%%writefile "{codigo_experimentos}"
from dataclasses import dataclass, replace
from hashlib import sha256
import json
from pathlib import Path
import sys
from time import perf_counter

import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from dados_etapa2 import NOMINAIS, carregar_treino, preparar_base, separar_divisoes
from tecnicas_etapa2 import transformar, medir


@dataclass(frozen=True)
class Config:
    representacao: str = "C"
    linhas: str = "todas"
    numericos: str = "mediana"
    nominais: str = "ausencia"
    codificacao: str = "onehot"
    escala: str = "zscore"
    balanceamento: str = "nenhum"
    reducao: str = "nenhuma"
    modelo: str = "arvore"


BASE = Config()
GERAL = {
    "referencia": BASE,
    "media": replace(BASE, numericos="media"),
    "mediana_com_indicador": replace(BASE, numericos="mediana_indicador"),
    "moda_nominal": replace(BASE, nominais="moda"),
    "onehot_raros": replace(BASE, codificacao="onehot_raros"),
    "inteiros_nominais": replace(BASE, codificacao="inteiros"),
    "representacao_A": replace(BASE, representacao="A"),
    "representacao_B": replace(BASE, representacao="B"),
    "sem_duplicatas": replace(BASE, linhas="sem_duplicatas"),
    "sem_conflitos": replace(BASE, linhas="sem_conflitos"),
    "sem_duplicatas_ou_conflitos": replace(BASE, linhas="sem_ambos"),
    "oversampling": replace(BASE, balanceamento="over"),
    "undersampling": replace(BASE, balanceamento="under"),
    "filtro_60": replace(BASE, reducao="filtro"),
    "pca_60": replace(BASE, reducao="pca"),
}
ESCALA = {
    f"rede_{escala}": replace(BASE, modelo="rede", escala=escala)
    for escala in ("zscore", "nenhuma", "minmax", "robusta")
}

CONFIRMACAO = {
  "arvore_referencia": {},
  "rede_referencia": {"modelo": "rede"},
  "arvore_sem_duplicatas": {"linhas": "sem_duplicatas"},
  "rede_sem_duplicatas": {"linhas": "sem_duplicatas", "modelo": "rede"},
  "arvore_pca_60": {"linhas": "sem_duplicatas", "reducao": "pca"},
  "rede_pca_60": {"linhas": "sem_duplicatas", "reducao": "pca", "modelo": "rede"},
  "arvore_oversampling": {"linhas": "sem_duplicatas", "balanceamento": "over"},
  "rede_oversampling": {"linhas": "sem_duplicatas", "balanceamento": "over", "modelo": "rede"},
  "arvore_onehot_raros": {"linhas": "sem_duplicatas", "codificacao": "onehot_raros"},
  "rede_onehot_raros": {"linhas": "sem_duplicatas", "codificacao": "onehot_raros", "modelo": "rede"},
  "arvore_melhores_medias": {"linhas": "sem_duplicatas", "numericos": "knn", "nominais": "moda", "codificacao": "inteiros"},
  "rede_melhores_medias": {"linhas": "sem_duplicatas", "numericos": "knn", "nominais": "moda", "codificacao": "inteiros", "modelo": "rede"},
  "rede_minmax": {"linhas": "sem_duplicatas", "escala": "minmax", "modelo": "rede"}
}


def executar(representacoes, y, divisoes, modo, configuracoes):
    linhas = []
    for fold, treino, validacao in divisoes:
        if (fold == 5) != (modo == "confirmar"):
            continue
        chave_anterior = None
        for nome, config in configuracoes.items():
            inicio = perf_counter()
            chave = (config.representacao, config.linhas, config.numericos,
                     config.nominais, config.codificacao, config.escala)
            if chave != chave_anterior:
                matrizes = transformar(representacoes[config.representacao], y, treino, validacao, config)
                chave_anterior = chave
            resultado = medir(*matrizes, y[validacao], config)
            linhas.append({"opcao": nome, "fold": fold, "modelo": config.modelo, **resultado,
                           "segundos": round(perf_counter() - inicio, 2)})
        print(f"{modo} — divisão {fold} concluída", flush=True)
    return linhas


def main():
    treino_csv, pasta = map(Path, sys.argv[1:])
    dados = carregar_treino(treino_csv)
    y = dados["class"].to_numpy()
    representacoes, grupos = preparar_base(dados)
    divisoes = separar_divisoes(representacoes, y, grupos)
    c = representacoes["C"]
    protocolo = {
        "linhas": len(dados),
        "colunas_originais": len(dados.columns),
        "atributos_por_versao": {versao: len(x.columns) for versao, x in representacoes.items()},
        "sha256": sha256(treino_csv.read_bytes()).hexdigest(),
        "grupos": int(grupos.nunique()),
        "categorias": {coluna: int(c[coluna].nunique()) for coluna in NOMINAIS if coluna in c},
        "divisoes": [
            {"fold": fold, "treino": len(tr), "validacao": len(va),
             "anomalias": int(np.count_nonzero(y[va] == "anomaly")), "grupos_em_comum": 0}
            for fold, tr, va in divisoes
        ],
    }
    combinacoes = {nome: replace(BASE, **opcoes) for nome, opcoes in CONFIRMACAO.items()}
    fases = {
        "geral": GERAL,
        "escala": ESCALA,
        "knn": {"knn_2000": replace(BASE, numericos="knn")},
        "combinar": {nome: config for nome, config in combinacoes.items() if not nome.endswith("_referencia")},
        "confirmar": combinacoes,
    }
    resultados = {}
    with threadpool_limits(limits=1):
        for modo, configuracoes in fases.items():
            resultados[modo] = executar(representacoes, y, divisoes, modo, configuracoes)
            pd.DataFrame(resultados[modo]).to_csv(pasta / f"{modo}.csv", index=False)
    (pasta / "resultados.json").write_text(
        json.dumps({"protocolo": protocolo, "resultados": resultados}, ensure_ascii=False),
        encoding="utf-8",
    )


if __name__ == "__main__":
    main()

In [ ]:
print("Executando as comparações com o treino oficial.", flush=True)
subprocess.run(
    [sys.executable, "-u", str(codigo_experimentos), str(treino_csv), str(trabalho)],
    check=True, env=ambiente,
)
dados_execucao = json.loads((trabalho / "resultados.json").read_text(encoding="utf-8"))
protocolo = dados_execucao["protocolo"]
resultados = dados_execucao["resultados"]
print(f"Ajustes concluídos: {sum(map(len, resultados.values()))}")

## Resultados

As tabelas são calculadas a partir desta execução. Nas quatro divisões iniciais, cada linha mostra a média e o desvio padrão amostral do F1-macro. A quinta divisão é apresentada separadamente.

In [ ]:
def exibir_tabela(colunas, linhas):
    cabecalho = " | ".join(colunas)
    separador = " | ".join(["---"] * len(colunas))
    corpo = [" | ".join(map(str, linha)) for linha in linhas]
    display(Markdown("\n".join(f"| {linha} |" for linha in (cabecalho, separador, *corpo))))

print(f"Treino oficial: {protocolo['linhas']:,} linhas e {protocolo['colunas_originais']} colunas.")
print("Atributos antes da codificação:", protocolo["atributos_por_versao"])
print("Categorias observadas na versão C, sem contar ausências:", protocolo["categorias"])
print("Grupos de entradas na versão C:", protocolo["grupos"])
exibir_tabela(
    ("Divisão", "Ajuste", "Validação", "Anomalias na validação", "Grupos em comum"),
    [(d["fold"], d["treino"], d["validacao"], d["anomalias"], d["grupos_em_comum"])
     for d in protocolo["divisoes"]],
)

In [ ]:
ROTULOS = {
    "referencia": "Árvore de referência",
    "media": "Imputação pela média",
    "mediana_com_indicador": "Mediana com indicador",
    "knn_2000": "Imputação KNN",
    "moda_nominal": "Moda para ausências nominais",
    "onehot_raros": "One-hot com serviços raros agrupados",
    "inteiros_nominais": "Códigos inteiros",
    "representacao_A": "Representação A",
    "representacao_B": "Representação B",
    "sem_duplicatas": "Retirada de duplicatas de mesmo rótulo",
    "sem_conflitos": "Retirada de grupos conflitantes",
    "sem_duplicatas_ou_conflitos": "Retirada de duplicatas e conflitos",
    "oversampling": "Sobreamostragem aleatória",
    "undersampling": "Subamostragem aleatória",
    "filtro_60": "Filtro univariado de 60 entradas",
    "pca_60": "PCA de 60 componentes",
    "rede_zscore": "Rede com Z-score",
    "rede_nenhuma": "Rede sem escala",
    "rede_minmax": "Rede com Min-Max",
    "rede_robusta": "Rede com escala robusta",
}

def resumo(registros, opcao):
    linhas = [r for r in registros if r["opcao"] == opcao]
    f1 = [r["f1_macro"] for r in linhas]
    colunas = sorted({r["colunas"] for r in linhas})
    dimensao = str(colunas[0]) if len(colunas) == 1 else f"{colunas[0]}–{colunas[-1]}"
    return (
        f"{mean(f1):.4f} ± {stdev(f1):.4f}",
        f"{mean(r['recall_anomaly'] for r in linhas):.2%}",
        f"{mean(r['precisao_anomaly'] for r in linhas):.2%}",
        dimensao,
        f"{mean(r['linhas_ajuste'] for r in linhas):.0f}",
    )

registros = resultados["geral"] + resultados["knn"] + resultados["escala"]
cabecalho_resumo = ("Opção", "F1-macro", "Recall de anomaly", "Precisão de anomaly", "Colunas", "Linhas no ajuste")
exibir_tabela(cabecalho_resumo, [(rotulo, *resumo(registros, opcao)) for opcao, rotulo in ROTULOS.items()])

In [ ]:
ROTULOS_COMBINACOES = {
    "arvore_referencia": "Árvore de referência",
    "rede_referencia": "Rede de referência",
    "arvore_sem_duplicatas": "Árvore com deduplicação",
    "rede_sem_duplicatas": "Rede com deduplicação",
    "arvore_pca_60": "Árvore com deduplicação e PCA",
    "rede_pca_60": "Rede com deduplicação e PCA",
    "arvore_oversampling": "Árvore com deduplicação e oversampling",
    "rede_oversampling": "Rede com deduplicação e oversampling",
    "arvore_onehot_raros": "Árvore com deduplicação e serviços raros agrupados",
    "rede_onehot_raros": "Rede com deduplicação e serviços raros agrupados",
    "arvore_melhores_medias": "Árvore com deduplicação, KNN, moda e códigos inteiros",
    "rede_melhores_medias": "Rede com deduplicação, KNN, moda e códigos inteiros",
    "rede_minmax": "Rede com deduplicação e Min-Max",
}
comparacao = resultados["combinar"] + [
    {**r, "opcao": "arvore_referencia"} for r in resultados["geral"] if r["opcao"] == "referencia"
] + [
    {**r, "opcao": "rede_referencia"} for r in resultados["escala"] if r["opcao"] == "rede_zscore"
]
reserva = {r["opcao"]: r for r in resultados["confirmar"]}
linhas = []
for opcao, rotulo in ROTULOS_COMBINACOES.items():
    f1, recall, _, _, _ = resumo(comparacao, opcao)
    medicao = reserva[opcao]
    linhas.append((rotulo, f1, recall, f"{medicao['f1_macro']:.4f}", f"{medicao['recall_anomaly']:.2%}"))
exibir_tabela(
    ("Combinação", "F1-macro médio", "Recall médio", "F1-macro na reserva", "Recall na reserva"),
    linhas,
)
print("CSV de resultados por divisão:", trabalho)

## Interpretação e escolhas para a Etapa 3

- **Representação e limpeza.** As versões A, B e C ficaram próximas. A versão C é a referência por ser mais simples. Nas configurações escolhidas, retiramos duplicatas de mesmo rótulo apenas no ajuste e preservamos grupos conflitantes, cuja retirada prejudicou o recall na comparação isolada.
- **Árvore.** A combinação de KNN numérico, moda para categorias ausentes e códigos inteiros teve o maior F1-macro entre as combinações com árvore. Ela usa a versão C, deduplicação, Z-score e todas as colunas resultantes, sem balanceamento.
- **Rede.** A melhor combinação avaliada para a rede usou versão C, deduplicação, mediana, categoria explícita para ausência, one-hot completo, Z-score e oversampling. A combinação com códigos inteiros que funcionou na árvore perdeu desempenho na rede. O oversampling também aumentou o recall da árvore, com queda no F1-macro.
- **Redução.** O PCA de 60 componentes preservou melhor o F1-macro do que o filtro, mas sua vantagem frente às entradas completas variou entre divisões. Por isso, as duas configurações propostas seguem sem redução adicional.

O KNN foi testado com referências amostradas e a rede com um orçamento curto de treinamento. As configurações são candidatas para a próxima etapa. A validação interna estima desempenho e mantém o teste oficial separado para a entrega final.

**Material da disciplina consultado.** Partes 1, 2, 3, 4, 6 e 7 dos slides de processamento, o notebook de codificação, imputação e balanceamento e o enunciado da competição.

**Apoio de IA.** Codex auxiliou na organização do código dos experimentos, na montagem do notebook e na verificação de reprodução. A equipe deve revisar a implementação e declarar esse apoio no relatório.